<a href="https://colab.research.google.com/github/alpercagan/spectralbridge/blob/main/notebooks/02_feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Week 2 Day 1: Feature Extraction - SpectralBridge Project
# ============================================================
# Extract 4 types of embeddings:
# 1. BEATs audio (self-supervised) - for SpectralBridge
# 2. DINOv2 image (self-supervised) - for SpectralBridge
# 3. CLAP audio (text-aligned) - for baseline comparison
# 4. CLIP image (text-aligned) - for baseline comparison

from google.colab import drive
from pathlib import Path
import numpy as np
from tqdm.notebook import tqdm
import torch

# Mount Drive
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/spectralbridge_data"

# Categories
SELECTED_CATEGORIES = [
    "playing piano", "playing acoustic guitar", "dog barking", "cat meowing",
    "helicopter", "typing on computer keyboard", "telephone bell ringing",
    "fire crackling", "ocean burbling", "thunder", "raining",
    "wind noise", "wind rustling leaves", "waterfall burbling",
    "hammering nails", "stream burbling",
]

# Verify data
audio_dir = Path(BASE_DIR) / "processed" / "audio"
image_dir = Path(BASE_DIR) / "processed" / "images"

total_audio = sum(len(list((audio_dir / cat).glob("*.wav")))
                 for cat in SELECTED_CATEGORIES if (audio_dir / cat).exists())
total_images = sum(len(list((image_dir / cat).glob("*.jpg")))
                  for cat in SELECTED_CATEGORIES if (image_dir / cat).exists())

print("="*60)
print("WEEK 2 DAY 1: FEATURE EXTRACTION")
print("="*60)
print(f"✓ Drive mounted: {BASE_DIR}")
print(f"✓ Categories: {len(SELECTED_CATEGORIES)}")
print(f"✓ Audio files: {total_audio}")
print(f"✓ Image files: {total_images}")
print("="*60)

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("="*60)

Mounted at /content/drive
WEEK 2 DAY 1: FEATURE EXTRACTION
✓ Drive mounted: /content/drive/MyDrive/spectralbridge_data
✓ Categories: 16
✓ Audio files: 1269
✓ Image files: 1269
✓ Device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB


In [2]:
# ============================================================
# Cell 2: Install Dependencies & Load Models
# ============================================================

print("Installing dependencies...")
print("="*60)

# Install required packages
!pip install -q timm transformers open_clip_torch torchaudio soundfile

print("✓ Packages installed")
print("="*60)
print("\nLoading models (this takes 2-3 minutes)...")
print("="*60)

import timm
import torch
import torchaudio
from transformers import ClapModel, ClapProcessor, CLIPModel, CLIPProcessor
from PIL import Image
import soundfile as sf

# 1. DINOv2 (Vision - Self-supervised)
print("Loading DINOv2...")
dinov2_model = timm.create_model(
    'vit_base_patch14_dinov2.lvd142m',
    pretrained=True,
    num_classes=0  # Remove classification head
).to(device).eval()
print("  ✓ DINOv2 loaded (768-d embeddings)")

# 2. CLIP (Vision - Text-aligned baseline)
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("  ✓ CLIP loaded (512-d embeddings)")

# 3. CLAP (Audio - Text-aligned baseline)
print("Loading CLAP...")
clap_model = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device).eval()
clap_processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
print("  ✓ CLAP loaded (512-d embeddings)")

# 4. BEATs will be loaded separately (requires manual download)
print("\nBEATs: Will download checkpoint separately")
print("="*60)
print("✓ 3/4 models loaded successfully")
print("="*60)

Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
✓ Packages installed

Loading models (this takes 2-3 minutes)...
Loading DINOv2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  ✓ DINOv2 loaded (768-d embeddings)
Loading CLIP...


config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

  ✓ CLAP loaded (512-d embeddings)

BEATs: Will download checkpoint separately
✓ 3/4 models loaded successfully
